In [0]:
# ============================================================
# Gold Layer - Delta Live Tables (DLT) Pipeline
# Spotify Analytics - Curated / Gold Layer
# Reads from Silver Delta tables, builds enriched gold views
# Deploy via: Databricks > Data Engineering > Delta Live Tables
# ============================================================

import dlt
from pyspark.sql import functions as F

print("Gold DLT pipeline loaded")

In [0]:
# ============================================================
# CELL 2: Gold - Top Tracks by Streams
# DLT Materialized View - Most played tracks
# ============================================================

@dlt.table(
    name="gold_top_tracks",
    comment="Top tracks ranked by total stream count",
    table_properties={"quality": "gold"}
)
def gold_top_tracks():
    streams = spark.read.table("hive_metastore.spotify_silver.fact_streams")
    tracks = spark.read.table("hive_metastore.spotify_silver.dim_tracks")
    artists = spark.read.table("hive_metastore.spotify_silver.dim_artists")

    return (
        streams
        .join(tracks, "track_id", "left")
        .join(artists, "artist_id", "left")
        .groupBy("track_id", "track_name", "artist_name")
        .agg(F.count("*").alias("total_streams"))
        .orderBy(F.desc("total_streams"))
        .limit(100)
    )

In [0]:
# ============================================================
# CELL 3: Gold - Top Artists by Streams
# DLT Materialized View - Most streamed artists
# ============================================================

@dlt.table(
    name="gold_top_artists",
    comment="Top artists ranked by total stream count",
    table_properties={"quality": "gold"}
)
def gold_top_artists():
    streams = spark.read.table("hive_metastore.spotify_silver.fact_streams")
    artists = spark.read.table("hive_metastore.spotify_silver.dim_artists")

    return (
        streams
        .join(artists, "artist_id", "left")
        .groupBy("artist_id", "artist_name")
        .agg(F.count("*").alias("total_streams"))
        .orderBy(F.desc("total_streams"))
        .limit(50)
    )

In [0]:
# ============================================================
# CELL 4: Gold - Top Albums by Streams
# DLT Materialized View - Most streamed albums
# ============================================================

@dlt.table(
    name="gold_top_albums",
    comment="Top albums ranked by total stream count",
    table_properties={"quality": "gold"}
)
def gold_top_albums():
    streams = spark.read.table("hive_metastore.spotify_silver.fact_streams")
    albums = spark.read.table("hive_metastore.spotify_silver.dim_albums")
    artists = spark.read.table("hive_metastore.spotify_silver.dim_artists")

    return (
        streams
        .join(albums, "album_id", "left")
        .join(artists, "artist_id", "left")
        .groupBy("album_id", "album_name", "artist_name")
        .agg(F.count("*").alias("total_streams"))
        .orderBy(F.desc("total_streams"))
        .limit(50)
    )

In [0]:
# ============================================================
# CELL 5: Gold - Monthly Streams Trend
# DLT Materialized View - Streaming trends over time
# ============================================================

@dlt.table(
    name="gold_monthly_streams",
    comment="Monthly stream counts aggregated by year and month",
    table_properties={"quality": "gold"}
)
def gold_monthly_streams():
    streams = spark.read.table("hive_metastore.spotify_silver.fact_streams")

    return (
        streams
        .withColumn("year", F.year("stream_date"))
        .withColumn("month", F.month("stream_date"))
        .groupBy("year", "month")
        .agg(F.count("*").alias("total_streams"))
        .orderBy("year", "month")
    )

In [0]:
# ============================================================
# CELL 6: Gold - Enriched User Listening History
# DLT Materialized View - Full enriched stream facts
# Joins all dimensions for a wide denormalized gold table
# ============================================================

@dlt.table(
    name="gold_user_listening_history",
    comment="Enriched user listening history with full track, artist, album, and user details",
    table_properties={"quality": "gold"}
)
def gold_user_listening_history():
    streams = spark.read.table("hive_metastore.spotify_silver.fact_streams")
    tracks = spark.read.table("hive_metastore.spotify_silver.dim_tracks")
    artists = spark.read.table("hive_metastore.spotify_silver.dim_artists")
    albums = spark.read.table("hive_metastore.spotify_silver.dim_albums")
    users = spark.read.table("hive_metastore.spotify_silver.dim_users")

    return (
        streams
        .join(tracks, "track_id", "left")
        .join(artists, "artist_id", "left")
        .join(albums, "album_id", "left")
        .join(users, "user_id", "left")
        .select(
            "stream_id",
            "user_id",
            F.col("user_name"),
            F.col("country"),
            "track_id",
            F.col("track_name"),
            "artist_id",
            F.col("artist_name"),
            "album_id",
            F.col("album_name"),
            "stream_date",
            F.col("duration_ms"),
            F.col("explicit")
        )
    )